# Building the FBI catalog

Writes `data/fbi_series_*.yaml` — the metadata the document store and
`series_catalog` read. The FBI is a live source, so these files describe series
that are fetched on request; nothing here lands in `time_series_source`.

**One file per group**, the convention every other source follows: BLS splits
by survey (23 files), BIS by dataflow (22), Clio by category (11). Here the
natural group is the **offence slug**, because that is the API's own request
dimension — so a re-harvest of one offence rewrites one file and the diffs stay
readable. Police employment is a separate family: annual rather than monthly,
agency-scoped, different facets, so it gets its own file.

    data/fbi_series_homicide.yaml          monthly, offences + clearances
    data/fbi_series_robbery.yaml           …one per offence, ten in all
    data/fbi_series_employment.yaml        annual, officers + civilians

**Cost.** One request per (scope, offence): 52 scopes × 10 offences = 520 calls
for the national and state catalog, about nine minutes at one a second and
inside the 1,000-per-hour allowance. Each agency adds one call per offence, so
choose cities deliberately.

There is no separate metadata fetch: each call returns the span, the
denominator, the coverage and the vintage alongside the observations.

In [1]:
import sys
from pathlib import Path

# Anchor on this notebook's own directory rather than on the working directory:
# VS Code can root a notebook at the workspace folder instead of at the file,
# and then a relative "../" resolves somewhere else entirely and the imports
# below fail with ModuleNotFoundError.
_here = Path.cwd()
_FBI = _here if (_here / "utils" / "mcp.py").exists() else _here / "notebooks" / "fbi"
for _path in (str(_FBI), str(_FBI.parents[1])):      # notebooks/fbi, then the meida root
    if _path not in sys.path:
        sys.path.insert(0, _path)

import asyncio
from collections import Counter

import yaml

from utils import catalog

print("scopes:", 1 + len(catalog.STATES), "| offences:", len(catalog.OFFENSE_NOTES))
print("calls for the national + state catalog:",
      (1 + len(catalog.STATES)) * len(catalog.OFFENSE_NOTES))

scopes: 52 | offences: 10
calls for the national + state catalog: 520


## The request vocabulary

Every entry's `retrieval` block names a tool and its arguments, and the rule is
that **`retrieval` minus `tool` is the argument list** — a consumer reads the
entry and calls the tool, with nothing to translate.

**Two tools, one per route:**

| tool | arguments | returns |
| --- | --- | --- |
| `fbi_offenses` | `offense`, `scope`, `measure`, `unit`, `start`, `end` | monthly counts, 1985– |
| `fbi_employment` | `ori`, `measure`, `start`, `end` | annual counts, 1985– |

**`offense` — ten values**, and they overlap. `violent-crime` is the total of
homicide, rape, robbery and aggravated assault; `property-crime` is the total
of burglary, larceny and motor vehicle theft, with **arson excluded** and
reported on its own. So the ten are four violent components, three property
components, two totals and arson. Summing all ten double-counts everything but
arson.

**`scope` — three forms, all one string:**

| form | example | count | where the values come from |
| --- | --- | --- | --- |
| the nation | `national` | 1 | a literal |
| **a two-letter state code** | `DC`, `TX` | 51 | `lookup/states` — 50 states and DC, no territories |
| an agency | `DCMPD0000` | 19,636 | `agency/byStateAbbr/{ST}`, in `data/raw/agencies.json` |

The client's `route()` resolves which of the three it has by shape, so the
catalog's `scope` facet and the tool's `scope` argument are the same string.

**`measure`** is `offenses` or `clearances` for crime, `officers` or
`civilians` for employment. It is part of `retrieval` because one call returns
both: without it the two entries for a scope would be indistinguishable.

**`unit`** is `count`. Rates are catalogued from the counts plus the population
the payload carries, rather than trusting the API's own rate.

In [2]:
from clients.fbi import OFFENSES

print("fbi_offenses — offense:")
totals = {"violent-crime", "property-crime"}
for offense in OFFENSES:
    kind = "total" if offense in totals else "component"
    print(f"   {offense:22} {kind:10} {catalog.OFFENSE_NOTES[offense][:60]}")

live_scopes = await catalog.scopes()
print(f"\nscope — {len(live_scopes)} geographic scopes from the API:")
print("   national, then state codes:", ", ".join(live_scopes[1:9]), "...")
print("   plus any ORI from the agency registry, e.g. DCMPD0000")

print("\nmeasure:  offenses | clearances   (fbi_offenses)")
print("          officers | civilians    (fbi_employment)")

fbi_offenses — offense:
   violent-crime          total      Murder and non-negligent manslaughter, rape, robbery and agg
   homicide               component  Murder and non-negligent manslaughter. Excludes negligent ma
   rape                   component  Revised definition from 2013; series crossing that year mix 
   robbery                component  Taking by force or threat of force.
   aggravated-assault     component  Assault with intent to cause severe bodily injury, usually i
   property-crime         total      Burglary, larceny-theft and motor vehicle theft combined; ar
   burglary               component  Unlawful entry to commit a felony or theft.
   larceny                component  Theft without force, violence or fraud.
   motor-vehicle-theft    component  Theft or attempted theft of a motor vehicle.
   arson                  component  Wilful burning. Reported less completely than the other offe



scope — 52 geographic scopes from the API:
   national, then state codes: AL, AK, AZ, AR, CA, CO, CT, DE ...
   plus any ORI from the agency registry, e.g. DCMPD0000

measure:  offenses | clearances   (fbi_offenses)
          officers | civilians    (fbi_employment)


## The cities

Agency scope gives crime *and* police staffing for the same department, which
is the pairing a city panel needs. Keep the list short and deliberate: each
city costs ten calls for offences plus one for employment.

In [3]:
CITIES = {
    "DCMPD0000": "Washington",
    "NY0303000": "New York City",
    "ILCPD0000": "Chicago",
    "CA0194200": "Los Angeles",
    "PAPEP0000": "Philadelphia",
}

# Confirm each ORI is in the registry before spending calls on it.
registry = catalog._agency_notes()
for ori, name in CITIES.items():
    record = registry.get(ori)
    print(f"  {ori}  {name:14} {'found: ' + record['agency_name'][:40] if record else 'NOT IN REGISTRY'}")

  DCMPD0000  Washington     found: Washington Police Department
  NY0303000  New York City  found: New York City Police Department
  ILCPD0000  Chicago        found: Chicago Police Department
  CA0194200  Los Angeles    found: Los Angeles Police Department
  PAPEP0000  Philadelphia   found: Philadelphia Police Department


## Harvest

National and every state, then the cities. Failures are collected rather than
raised, so one bad scope does not lose the run; anything that failed lands in
`data/harvest_failures.json`.

In [4]:
entries = await catalog.harvest(scopes=("national", *catalog.STATES))
print(f"national + state entries: {len(entries):,}")

national + state entries: 924


In [5]:
city_entries = await catalog.harvest(scopes=tuple(CITIES))
employment = await catalog.harvest_employment(CITIES)
print(f"city offence entries: {len(city_entries)}")
print(f"employment entries:   {len(employment)}")

city offence entries: 100
employment entries:   10


## Write

One file per group, sorted by `series_id` so successive exports diff cleanly.

In [6]:
report = catalog.export(entries + city_entries + employment)
print(report)

for path in sorted(catalog.DATA_DIR.glob("fbi_series_*.yaml")):
    doc = yaml.safe_load(path.read_text())
    print(f"  {path.name:34} {doc['series_count']:5} series")

{'entries': 1034, 'files': 11}
  fbi_series_aggravated_assault.yaml    86 series
  fbi_series_arson.yaml                114 series
  fbi_series_burglary.yaml             114 series


  fbi_series_employment.yaml            10 series
  fbi_series_homicide.yaml             114 series
  fbi_series_larceny.yaml              114 series


  fbi_series_motor_vehicle_theft.yaml   114 series
  fbi_series_property_crime.yaml        88 series
  fbi_series_rape.yaml                 112 series
  fbi_series_robbery.yaml               54 series


  fbi_series_violent_crime.yaml        114 series


## What one entry holds

The fields come from four places: the payload (span, coverage, vintage,
geography), the request (id, title, facets, retrieval), the agency registry
(type, state, NIBRS start date) and hand-written definitions. Only
`description` is still missing — generated later by `db_import.descriptions`,
the same bucketing CDC uses, over about twenty buckets rather than CDC's
two and a half thousand series.

In [7]:
sample = yaml.safe_load((catalog.DATA_DIR / "fbi_series_homicide.yaml").read_text())
entry = next(e for e in sample["series"] if e["facets"]["scope"] == "DCMPD0000"
             and e["facets"]["measure"] == "clearances")
print(yaml.safe_dump(entry, sort_keys=False, allow_unicode=True, width=86))

series_id: fbi/homicide/DCMPD0000/clearances
concept: homicide
title: Homicide clearances, Washington Police Department, monthly
unit: count
frequency: Monthly
provisional: false
is_active: true
facets:
  geography: Washington Police Department
  scope: DCMPD0000
  offense: homicide
  measure: clearances
  agency_type: City
  state: DC
  county: NOT SPECIFIED
observation_start: '1985-01-01'
observation_end: '2024-12-01'
definition: Murder and non-negligent manslaughter. Excludes negligent manslaughter, justifiable
  homicide and attempts. Offences cleared by arrest or exceptional means. Divided by offences
  this is the clearance rate, the usual measure of enforcement intensity.
coverage_mean_percent: 100.0
coverage_min_percent: 100.0
sources:
- kind: fbi_cde
  endpoint: summarized/DCMPD0000/homicide
  vintage: 09/2026
  refreshed: 09/15/2026
retrieval:
  tool: fbi_offenses
  offense: homicide
  scope: DCMPD0000
  measure: clearances
  unit: count
nibrs_start_date: '2021-08-01'



## Coverage, as a catalog field

`coverage_min_percent` is what makes an entry honest about 2021: the FBI
retired the Summary Reporting System that January in favour of NIBRS, agencies
that had not converted dropped out, and the counting rule changed at the same
time. A series whose minimum coverage stays near 100 crosses that year cleanly;
one that dips to 74 does not.

In [8]:
national = [e for e in entries if e["facets"]["scope"] == "national"]
worst = sorted(national, key=lambda e: e["coverage_min_percent"] or 100)[:5]
for e in worst:
    print(f"  {e['series_id']:44} min coverage {e['coverage_min_percent']:5.1f}%"
          f"  mean {e['coverage_mean_percent']:5.1f}%")

states = [e for e in entries if e["facets"]["scope"] != "national"
          and e["facets"]["measure"] == "offenses" and e["concept"] == "homicide"]
clean = [e for e in states if (e["coverage_min_percent"] or 0) >= 95]
print(f"\nstates whose homicide coverage never fell below 95%: {len(clean)} of {len(states)}")

  fbi/violent-crime/national/offenses          min coverage  74.1%  mean  93.7%
  fbi/violent-crime/national/clearances        min coverage  74.1%  mean  93.7%
  fbi/homicide/national/offenses               min coverage  74.1%  mean  93.7%
  fbi/homicide/national/clearances             min coverage  74.1%  mean  93.7%
  fbi/rape/national/offenses                   min coverage  74.1%  mean  93.7%

states whose homicide coverage never fell below 95%: 5 of 51


## Next

`db_import.load_catalog.load(DATA_DIR, "fbi")` puts these rows in
`series_catalog`, where `retrieval` routes each one to the live MCP tool rather
than to stored observations — the same split CDC uses between its Socrata
series and its WONDER ones.